In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# DecisionTreeRegressor

In [ ]:
train= pd.read_csv('/kaggle/input/house-prices-advanced-regression-techniques/train.csv')
test= pd.read_csv('/kaggle/input/house-prices-advanced-regression-techniques/test.csv')

In [ ]:
train.columns

In [ ]:
train.isnull().sum()

In [ ]:
test.dtypes

# ستون SalePrice = output_train

In [ ]:
output_train= train['SalePrice']
output_train

# input_train → تمام ستون‌ها به جز SalePrice و Id

In [ ]:
input_train= train.drop(['SalePrice', 'Id'], axis=1)
input_train

# cat_train → فقط ستون‌های categorical

In [ ]:
cat_train= input_train.select_dtypes(include='object')
cat_train

# num_train → فقط ستون‌های numeric

In [ ]:
num_train= input_train.select_dtypes(include=['int64', 'float64'])
num_train

# cat_input_filled_df → نسخه‌ی بدون داده های گمشده از داده‌های دسته‌ای

In [ ]:
from sklearn.impute import SimpleImputer
cat_imputer= SimpleImputer(strategy='most_frequent')
cat_input_filled= cat_imputer.fit_transform(cat_train)
cat_input_filled_df= pd.DataFrame(cat_input_filled, columns= cat_train.columns)

# num_input_filled_df → نسخه‌ی بدون داده های گمشده از داده‌های عددی

In [ ]:
num_inputer= SimpleImputer(strategy='median')
num_input_filled= num_inputer.fit_transform(num_train)
num_input_filled_df=pd.DataFrame(num_input_filled, columns=num_train.columns)

# filling missing values in test data

In [ ]:
input_test = test.drop(['Id'], axis=1)
cat_test = input_test.select_dtypes(include='object')
num_test = input_test.select_dtypes(include=['int64', 'float64'])

In [ ]:
num_test_fill= num_inputer.transform(num_test)
cat_test_fill= cat_imputer.transform(cat_test)
num_test_fill_df= pd.DataFrame(num_test_fill, columns=num_test.columns)
cat_test_fill_df = pd.DataFrame(cat_test_fill, columns=cat_test.columns, index=cat_test.index)
cat_test_fill_df

# factorization categorical

In [ ]:
cat_train_fact = pd.DataFrame()
cat_test_fact = pd.DataFrame()

# حلقه روی همه‌ی ستون‌های متنی
for col in cat_input_filled_df.columns:
    # تبدیل ستون متنی به عدد روی داده‌های train
    cat_train_fact[col], uniques = pd.factorize(cat_input_filled_df[col])
    
    # اعمال همان نگاشت روی داده‌های test
    cat_test_fact[col] = cat_test_fill_df[col].apply(
        lambda x: np.where(uniques == x)[0][0] if x in uniques else -1
    )

In [ ]:
final_train = pd.concat([num_input_filled_df, cat_train_fact], axis=1)
final_test = pd.concat([num_test_fill_df, cat_test_fact], axis=1)

In [ ]:
from sklearn.tree import DecisionTreeRegressor
model= DecisionTreeRegressor(random_state=42,max_depth=8)
from sklearn.model_selection import train_test_split 

In [ ]:
X_train, X_val, y_train, y_val = train_test_split(final_train, output_train, test_size=0.2, random_state=42)
model.fit(X_train, y_train)
predictions_val = model.predict(X_val)

In [ ]:
compare_y= pd.DataFrame({'test_y': y_val, 'predicted': predictions_val})
compare_y

In [ ]:
from sklearn.metrics import r2_score , mean_squared_error
R2= r2_score(y_val, predictions_val)
R2

In [ ]:
mse= mean_squared_error(y_val, predictions_val)
mse

In [ ]:
print("MSE:", mse)
print("R2:", R2)

In [ ]:
from sklearn.tree import plot_tree
import matplotlib.pyplot as plt


plt.figure(figsize=(20,10))
plot_tree(model, 
          feature_names=final_train.columns, 
          filled=True,       # رنگ بندی برگ‌ها
          rounded=True,      # گوشه‌های گرد
          fontsize=10)
plt.show()

In [ ]:
model.fit(final_train, output_train)

predictions_test = model.predict(final_test)
submission = pd.DataFrame({'Id': test['Id'], 'SalePrice': predictions_test})
submission.to_csv('submission_decision_tree.csv', index=False)
print("✅ Decision Tree submission آماده شد")